In [22]:
from scipy.io import arff
import pandas as pd
import numpy as np
from scipy.io import arff
from collections import Counter
from Class import *
from pathlib import Path
import joblib
import os
from typing import Tuple, Union
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.utils import shuffle
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

In [49]:
# ordering methods

def hardness(weights):
    return np.argsort(weights)

def rev_hardness(weights):
    return np.argsort(weights)[::-1]

def hardness_partition(weights):
    easy_idx = np.where(weights <= np.average(weights))[0]
    hard_idx = np.where(weights > np.average(weights))[0]

    if len(easy_idx) == 0 or len(hard_idx) == 0:
        return np.random.permutation(len(weights))

    np.random.shuffle(easy_idx)
    np.random.shuffle(hard_idx)

    return np.concatenate([easy_idx, hard_idx])

def rev_hardness_partition(weights):
    easy_idx = np.where(weights <= np.average(weights))[0]
    hard_idx = np.where(weights > np.average(weights))[0]

    if len(easy_idx) == 0 or len(hard_idx) == 0:
        return np.random.permutation(len(weights))

    np.random.shuffle(easy_idx)
    np.random.shuffle(hard_idx)

    return np.concatenate([hard_idx, easy_idx])

def three_level(weights):
    p33 = np.percentile(weights, 33)
    p66 = np.percentile(weights, 66)

    easy = np.where(weights <= p33)[0]
    mid  = np.where((weights > p33) & (weights <= p66))[0]
    hard = np.where(weights > p66)[0]

    if len(easy) == 0 or len(mid) == 0 or len(hard) == 0:
        order = np.argsort(weights)
        return order

    np.random.shuffle(easy)
    np.random.shuffle(mid)
    np.random.shuffle(hard)

    return np.concatenate([easy, mid, hard])

def probabilistic(weights, T=0.5):
    weights = np.asarray(weights, dtype=float)

    if len(weights) == 0:
        return np.array([], dtype=int)

    if T <= 0:
        raise ValueError("T must be positive")

    if not np.all(np.isfinite(weights)):
        raise ValueError("Weights must contain only finite values")

    scores = weights / T
    scores -= scores.max()

    p = np.exp(scores)
    p = np.maximum(p, 1e-12)
    p /= p.sum()

    return np.random.choice(
        len(weights),
        size=len(weights),
        replace=False,
        p=p
    )


def rev_probabilistic(weights, T=0.5):
    weights = np.asarray(weights, dtype=float)

    if len(weights) == 0:
        return np.array([], dtype=int)

    if T <= 0:
        raise ValueError("T must be positive")

    if not np.all(np.isfinite(weights)):
        raise ValueError("Weights must contain only finite values")

    scores = -weights / T
    scores -= scores.max()

    p = np.exp(scores)
    p = np.maximum(p, 1e-12)
    p /= p.sum()

    return np.random.choice(
        len(weights),
        size=len(weights),
        replace=False,
        p=p
    )

def class_aware(weights, labels):
    final_idx = []

    classes = np.unique(labels)
    class_indices = {}

    for c in classes:
        c_idx = np.where(labels == c)[0]
        c_idx = c_idx[np.argsort(weights[c_idx])]
        class_indices[c] = list(c_idx)

    while any(len(v) > 0 for v in class_indices.values()):
        for c in classes:
            if class_indices[c]:
                final_idx.append(class_indices[c].pop(0))

    return np.array(final_idx)

ordering_methods = {
    "hardness" : hardness,
    "rev_hardness" : rev_hardness,
    "hardness_partition" : hardness_partition,
    "rev_hardness_partition" : rev_hardness_partition,
    "three_level" : three_level,
    "probabilistic" : probabilistic,
    "rev_probabilistic" : rev_probabilistic,
    # "class_aware" : class_aware
}

In [24]:
# weighting methods

def harmfulness(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    D = pairwise_distances(X)
    harm = np.zeros(len(X))

    for i in range(len(X)):
        enemy_idx = np.where(y != y[i])[0]
        nearest_enemy = enemy_idx[
            np.argmin(D[i, enemy_idx])
        ]
        harm[nearest_enemy] += 1

    class_sizes = np.array([
        np.sum(y == y[i])
        for i in range(len(y))
    ])

    return harm / class_sizes

def usefulness(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    D = pairwise_distances(X)
    use = np.zeros(len(X))

    for i in range(len(X)):
        same_mask = y == y[i]
        other_mask = y != y[i]
        same_mask[i] = False
        nearest_other = np.min(D[i, other_mask])
        ls_idx = np.where(same_mask & (D[i] < nearest_other))[0]
        use[ls_idx] += 1

    return use

def LSR(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    D = pairwise_distances(X)
    weights = np.zeros(len(X))
    for i in range(len(X)):
        same_mask = y == y[i]
        other_mask = y != y[i]
        same_mask[i] = False

        if np.sum(same_mask) == 0:
            continue

        nearest_other = np.min(D[i, other_mask])
        farthest_same = np.max(D[i, same_mask])
        value = nearest_other / (farthest_same + 1e-10)
        weights[i] = 1 - np.minimum(1.0, value)

    return weights

def LSC(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    D = pairwise_distances(X)
    weights = np.zeros(len(X))
    for i in range(len(X)):
        same_mask = y == y[i]
        other_mask = y != y[i]
        same_mask[i] = False
        n = np.sum(same_mask)

        if n == 0:
            continue

        nearest_other = np.min(D[i, other_mask])
        LS = np.sum(D[i, same_mask] < nearest_other)
        weights[i] = 1 - LS / n

    return weights

def N2(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    D = pairwise_distances(X)
    weights = np.zeros(len(X))
    for i in range(len(X)):
        same_mask = y == y[i]
        other_mask = y != y[i]
        same_mask[i] = False
        same_dist = D[i, same_mask]
        other_dist = D[i, other_mask]

        if len(same_dist) == 0 or len(other_dist) == 0:
            weights[i] = 0
            continue

        nearest_same = np.min(same_dist)
        nearest_other = np.min(other_dist)
        intrainter = nearest_same / (nearest_other + 1e-10)
        weights[i] = 1 - 1 / (intrainter + 1)

    return weights  

def F1(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes = np.unique(y)
    n_samples, n_features = X.shape
    class_mins = []
    class_maxs = []

    for c in classes:
        X_c = X[y == c]
        class_mins.append(np.min(X_c, axis=0))
        class_maxs.append(np.max(X_c, axis=0))

    class_mins = np.array(class_mins)
    class_maxs = np.array(class_maxs)

    lower = np.max(class_mins, axis=0)
    upper = np.min(class_maxs, axis=0)

    inside = (X >= lower) & (X <= upper)
    weights = np.sum(inside, axis=1) / n_features

    return weights

def mahalanobis_weights(X):
    X = np.asarray(X, dtype=float)
    mu = np.mean(X, axis=0)
    cov = np.cov(X, rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-8
    inv_cov = np.linalg.inv(cov)
    diff = X - mu
    dist = np.sqrt(
        np.sum((diff @ inv_cov) * diff, axis=1)
    )
    return dist

def gaussian_class_margin(X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)

    if y.ndim == 2:
        y = np.argmax(y, axis=1)

    classes = np.unique(y)
    n = len(X)

    class_likelihoods = np.zeros((n, len(classes)))
    class_to_col = {c: i for i, c in enumerate(classes)}

    for c in classes:
        X_c = X[y == c]
        mu = np.mean(X_c, axis=0)
        sigma = np.std(X_c, axis=0) + 1e-10
        col = class_to_col[c]
        class_likelihoods[:, col] = -0.5 * np.sum(
            np.log(2 * np.pi * sigma**2)
            + ((X - mu) ** 2) / (sigma**2),
            axis=1
        )

    y_col = np.array([class_to_col[label] for label in y])
    real_cl = class_likelihoods[np.arange(n), y_col]
    other_cls = class_likelihoods.copy()
    other_cls[np.arange(n), y_col] = -np.inf

    max_other_cl = np.max(other_cls, axis=1)
    weights = real_cl - max_other_cl
    weights -= weights.min()
    if weights.max() > 0:
        weights /= weights.max()

    return weights

def gaussian_class_likelihood(X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)
    if y.ndim == 2:
        y = np.argmax(y, axis=1)

    weights = np.zeros(len(X))
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        X_c = X[idx]
        mu = np.mean(X_c, axis=0)
        sigma = np.std(X_c, axis=0) + 1e-10
        likelihood = np.ones(len(idx))
        for j in range(X.shape[1]):
            p = (
                1 / (np.sqrt(2 * np.pi) * sigma[j])
            ) * np.exp(
                -((X_c[:, j] - mu[j]) ** 2)
                / (2 * sigma[j] ** 2)
            )
            likelihood *= p

        weights[idx] = likelihood
    weights -= weights.min()
    if weights.max() > 0:
        weights /= weights.max()
    return weights

def tree_depth(X, rules_if):
    weights = np.zeros(len(X))

    for rule_if in rules_if:
        covered = racer._covered(X, rule_if)
        n_ones = np.sum(rule_if.astype(int))
        weights[covered] = n_ones

    return weights

def DS_fitness(X, rules_if, rules_fitness):
    weights = np.zeros(len(X))

    rules_fitness = np.array(rules_fitness)
    max_fit = rules_fitness.max()

    for rule_if, fitness in zip(rules_if, rules_fitness):
        covered = racer._covered(X, rule_if)

        if max_fit != 0:
            weights[covered] = fitness / max_fit

    return weights

def DCP(X, y, rules_if, rules_then):
    if y.ndim == 2:
        y_class = np.argmax(y, axis=1)
    else:
        y_class = y.astype(int)
    
    weights = np.zeros(len(X))
    if rules_then.ndim == 2:
        rules_then_class = np.argmax(rules_then, axis=1)
    else:
        rules_then_class = rules_then

    for rule_if, rule_then in zip(rules_if, rules_then_class):
        covered = racer._covered(X, rule_if)
        rule_cov = np.sum(covered)

        if rule_cov == 0:
            continue

        same_class_cov = np.sum(covered & (y_class == rule_then))
        weights[covered] = same_class_cov / rule_cov

    return weights

def DS(X, rules_if):
    weights = np.zeros(len(X))
    rule_coverages = []

    for rule_if in rules_if:
        covered = racer._covered(X, rule_if)
        rule_coverages.append(np.sum(covered))

    rule_coverages = np.array(rule_coverages)
    max_cov = rule_coverages.max()
    for rule_if, cov in zip(rules_if, rule_coverages):
        covered = racer._covered(X, rule_if)
        idx = np.where(covered)[0]
        weights[idx] = cov / max_cov
    
    return weights

def knn_disagreeing(X, Y, k=5):
    n_samples = X.shape[0]
    if Y.ndim > 1:
        Y_labels = np.argmax(Y, axis=1)
    else:
        Y_labels = Y.astype(int)

    knn = NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, indices = knn.kneighbors(X)
    weights = np.zeros(n_samples)
    for i in range(n_samples):
        neighbor_indices = indices[i, 1:]
        neighbor_labels = Y_labels[neighbor_indices]
        current_label = Y_labels[i]    
        weights[i] = np.mean(neighbor_labels != current_label)
    
    return weights / np.sum(weights)

def knn_weighting(X, Y, k=5):
    n_samples = X.shape[0]
    if Y.ndim > 1:
        Y_labels = np.argmax(Y, axis=1)
    else:
        Y_labels = Y.astype(int)
    
    knn = NearestNeighbors(n_neighbors=k+1).fit(X)
    distances, indices = knn.kneighbors(X)
    weights = np.zeros(n_samples)
    for i in range(n_samples):
        neighbor_indices = indices[i, 1:]
        neighbor_labels = Y_labels[neighbor_indices]
        current_label = Y_labels[i]
        local_counts = Counter(neighbor_labels)
        local_majority = local_counts.most_common(1)[0][0]
        scores = []
        for n_label in neighbor_labels:
            is_right = (n_label == current_label)
            is_local_maj = (n_label == local_majority)
            if is_right and is_local_maj:
                scores.append(1.00)
            elif is_right and not is_local_maj:
                scores.append(0.75)
            elif not is_right and not is_local_maj:
                scores.append(0.50)
            else:
                scores.append(0.25)
        
        weights[i] = np.mean(scores)
    
    return weights / np.sum(weights)

def weak_strong(X, y, maxalpha, minalpha):
    y_true = y.astype(int)[:,1]
    weak_racer = ROPAC(alpha=maxalpha, gamma=0.6, suppress_warnings=True)
    weights = np.ones(len(X), dtype=float)
    if maxalpha < minalpha:
        minalpha = 0.5
    while(weak_racer._alpha > minalpha):
        weak_racer.fit(X, y)
        weak_racer.reduceRules()
        y_pred = weak_racer.predict(X)
        mask = y_true != y_pred

        weights[mask] += 0.25
        weak_racer._alpha -= 0.01
        weak_racer._beta = 1 - weak_racer._alpha

    return weights / weights.sum()

# Data Cleaning

In [25]:
datasetname = "Toxicity"
data_file = "../data/{0}.csv".format(datasetname)

In [26]:
# csv
df = pd.read_csv(
        data_file,
        header= None,
        sep=",",
        na_values="?",
        skipinitialspace=True, 
        # encoding="utf-16"
    )

In [6]:
# xlsx
df = pd.read_excel(data_file, sheet_name=0)

In [29]:
df

,0,1,2,3,4,5,6,7,8,9,...,1194,1195,1196,1197,1198,1199,1200,1201,1202,1203
0,0.0908,0,0.0075,0.0173,0,0,-0.0436,0.0409,0,0.1368,...,0,0,0.178,1.5488,0.0088,0,0,-0.0868,0,NonToxic
1,0.0213,0,0.1144,-0.041,0,0,0.1231,-0.0316,0,0.1318,...,8.866,19.3525,0.1739,1.3718,0.0048,2,0,-0.081,0.25,NonToxic
2,0.0018,0,-0.0156,-0.0765,2,0,-0.1138,-0.1791,0,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0,-0.1004,0,NonToxic
3,-0.0251,0,-0.0064,-0.0894,3,0,-0.0747,-0.1151,0,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0,-0.101,0,NonToxic
4,0.0135,0,0.0424,-0.0353,0,0,-0.0638,0.0307,0,0.0306,...,12.324,19.7486,0.1789,1.4495,0.012,2,0,-0.1071,0,NonToxic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,-0.096,0,-0.0478,-0.084,2,0,-0.0739,-0.2315,1.566,-0.1133,...,2.569,12.0174,0.1648,0.971,0.0049,1,0,-0.0952,0,NonToxic
167,-0.0064,1,-0.1222,0.0013,1,0,-0.1873,-0.2181,5.5404,-0.0757,...,10.786,6.4871,0.1805,1.2298,0.0127,1,0,-0.086,0,NonToxic
168,0.0096,2,-0.1846,0.0058,1,0,-0.1293,-0.0979,5.3976,0.0409,...,4.993,19.2864,0.2089,1.1245,0.0093,1,0,-0.0927,0,NonToxic
169,-0.0736,2,-0.1267,-0.0345,2,0.5346,-0.0361,0.0151,5.519,-0.1025,...,10.7504,19.4989,0.1944,1.2256,0.0167,1,0,-0.1129,0,Toxic


In [ ]:
# for feature vector datasets
import re

def load_sparse_data(file_path):
    rows = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip().strip("{}")

            pairs = line.split(",")

            row = {}

            for pair in pairs:
                idx, value = pair.strip().split()
                row[int(idx)] = float(value)

            rows.append(row)

    # find number of attributes
    n_features = max(
        max(row.keys()) for row in rows
    ) + 1

    # create dense matrix
    data = np.zeros((len(rows), n_features))

    for i, row in enumerate(rows):
        for col, value in row.items():
            data[i, col] = value

    df = pd.DataFrame(data)

    return data, df

data, df = load_sparse_data(data_file)

In [ ]:
# if target column needs swapping
cols = list(df.columns)
cols[1], cols[2] = cols[2], cols[1]
df = df[cols]

In [28]:
# remove first row if it contains column names
df = df.iloc[1:].reset_index(drop=True)

In [25]:
# remove first column if it contains ID
df = df.iloc[:, 1:].reset_index(drop=True)

In [30]:
dataTypes = []
for col in df.columns:
    if df[col].dropna().map(type).eq(bool).all():
        dataTypes.append("nominal")
    
    else:
        numeric = pd.to_numeric(df[col], errors="coerce")

        if numeric.notna().sum() == df[col].notna().sum():
            df[col] = numeric
            dataTypes.append("numeric")
        else:
            dataTypes.append("nominal")
dataTypes

['numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'numeric',
 'nu

In [31]:
# based on target column placement

classIndex=len(df.columns) - 1
# classIndex=0

In [42]:
# prepare Data
from sklearn.impute import SimpleImputer

dataNum = df.copy()
dataSet = dataNum.to_numpy(dtype=object)

# Boolian Dataset
preprocessor = ROPACPreprocesser()
X, y = preprocessor.fit_transform(dataSet.copy(), dataTypes, classIndex)

X = np.asarray(X, dtype=bool)
y = np.asarray(y, dtype=bool)

# Numeric dataset
catAttrIndex = [
    i for i, t in enumerate(dataTypes)
    if t not in ["numeric", "integer", "real"]
    and i != classIndex
]
if catAttrIndex:
    for i in catAttrIndex:
        dataNum.iloc[:, i] = dataNum.iloc[:, i].astype("category").cat.codes

X_Num = dataNum.drop(
    columns=dataNum.columns[classIndex]
).to_numpy(dtype=float)

# Use exactly the same target encoding as ROPAC
y_Num = np.argmax(y, axis=1)

# Shuffle the data
rng = np.random.default_rng()
idx = rng.permutation(len(X))

X = X[idx]
y = y[idx]

X_Num = X_Num[idx]
y_Num = y_Num[idx]


imputer = SimpleImputer(strategy="mean")
X_Num = imputer.fit_transform(X_Num)

kf = KFold(n_splits=10, shuffle=False)

# Finding best Alpha

In [ ]:
# best alpha
racer = ROPAC(gamma=0.6, suppress_warnings=True)
max=0
min=1
while True:
    score = 0
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        Y_train, Y_test = y[train_index], y[test_index]

        racer.fit(X_train, Y_train)
        racer.reduceRules()
        score += racer.score(X_test, Y_test)

    if float(score/10) > max :
        max = float(score/10)
        maxalpha = racer._alpha

    if float(score/10) < min :
        min = float(score/10)
        minalpha = racer._alpha
    
    racer._alpha -= 0.01
    racer._beta = 1 - racer._alpha
    if racer._alpha<0.5:
        break

max, maxalpha, min, minalpha

In [33]:
# for large datasets
maxalpha = 0.99
minalpha = 0.50

# Train

In [34]:
def summarize_results(fold_accuracies):
    results = {}

    print(f"\n{'=' * 17} Final Results {'=' * 17}")

    for model_name, scores in fold_accuracies.items():
        scores = np.asarray(scores, dtype=np.float64)

        valid_scores = scores[~np.isnan(scores)]

        if len(valid_scores) == 0:
            mean_accuracy = np.nan
            std_accuracy = np.nan
        else:
            mean_accuracy = np.mean(valid_scores)
            std_accuracy = np.std(valid_scores)

        results[model_name] = {
            "mean_accuracy": float(mean_accuracy),
            "std_accuracy": float(std_accuracy),
            "fold_accuracies": scores.tolist()
        }

        if np.isnan(mean_accuracy):
            print(f"{model_name:<20}: failed")
        else:
            print(
                f"{model_name:<20}: "
                f"{mean_accuracy * 100:.2f}% "
                f"± {std_accuracy * 100:.2f}%"
            )

    return results

In [43]:
# best kf
racerAvgAcc = 0
fold_results = {
    ordering_name: {
        "DCP": []
    }
    for ordering_name in ordering_methods
}

for fold_number, (train_index, test_index) in enumerate(
    kf.split(X),
    start=1
):

    print(f"\n{'=' * 17} Fold {fold_number} {'=' * 17}")

    # --------------------------------------------------------
    # 1. Original train/test split
    # --------------------------------------------------------

    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = y[train_index], y[test_index]

    X_train_Num = X_Num[train_index]
    Y_train_Num = y_Num[train_index]
    Y_test_Num = y_Num[test_index]


    # ========================================================
    # 2. Standard RACER baseline
    # ========================================================

    racer = ROPAC(
        alpha=maxalpha,
        gamma=0.6,
        suppress_warnings=True
    )

    racer.fit(X_train, Y_train)
    racer.reduceRules()

    baseline_score = racer.score(X_test, Y_test)

    racerAvgAcc += baseline_score

    print(
        f"\nStandard RACER Fold Accuracy: "
        f"{baseline_score:.4f}"
    )

    weighting_racer = ROPAC(
        alpha=maxalpha - 0.15,
        gamma=0.6,
        suppress_warnings=True
    )

    weighting_racer.fit(X_train, Y_train)
    weighting_racer.reduceRules()

    weights = DCP(
        X_train,
        Y_train,
        weighting_racer._final_rules_if,
        weighting_racer._final_rules_then
    )

    for ordering_name, ordering_func in ordering_methods.items():

        idx = ordering_func(weights)

        X_train_sorted = X_train[idx]
        Y_train_sorted = Y_train[idx]
        weights_sorted = weights[idx]

        racer_reordered = ROPAC(
            alpha=maxalpha,
            gamma=0.6,
            suppress_warnings=True
        )
        racer_reordered.fit(
            X_train_sorted,
            Y_train_sorted
        )
        racer_reordered.reduceRules()
        y_pred = racer_reordered.predict(X_test)
        reordered_score = racer_reordered.score(X_test, Y_test)

        fold_results[ordering_name]["DCP"].append(
            reordered_score
        )

        print(f"| Accuracy: {reordered_score:.4f}")

print(f"\n{'=' * 17} Standard RACER {'=' * 17}")

baseline_mean = racerAvgAcc / kf.get_n_splits()

print(
    f"Standard RACER: "
    f"{baseline_mean * 100:.2f}%"
)

final_summary = {
    "baseline": {
        "mean_accuracy": float(baseline_mean)
    },
    "weighting_results": {}
}

weighting_scores = {}

for ordering_name in ordering_methods:

    weighting_scores[ordering_name] = (
        fold_results[ordering_name]["DCP"]
    )

# Average over folds for each ordering
weighting_summary = summarize_results(
    weighting_scores
)


================= Fold 1 =================
Labels for these instances will be determined by a closest match algorithm.

Standard RACER Fold Accuracy: 0.7222
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
| Accuracy: 0.8333
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
| Accuracy: 1.0000
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
| Accuracy: 0.8889
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
| Accuracy: 0.9444
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
| Accu

In [ ]:
# Train All methods
save_dir = os.path.join("../saved_results", datasetname)

# ============================================================
# K-Fold Experiment
# ============================================================

racerAvgAcc = 0
fold_accuracies = {
    ordering_name: {
        weighting_name: []
        for weighting_name in [
            "harmfulness",
            "usefulness",
            "LSR",
            "LSC",
            "N2",
            "F1",
            "mahalanobis_weights",
            "gaussian_class_margin",
            "gaussian_class_likelihood",
            "knn_disagreeing",
            "knn_weighting",
            "weak_strong",
            "tree_depth",
            "DS_fitness",
            "DCP",
            "DS"
        ]
    }
    for ordering_name in ordering_methods
}

# Directory for saved fold results
os.makedirs(save_dir, exist_ok=True)

for fold_number, (train_index, test_index) in enumerate(
    kf.split(X),
    start=1
):

    print(f"\n{'=' * 17} Fold {fold_number} {'=' * 17}")

    # --------------------------------------------------------
    # 1. Original train/test split
    # --------------------------------------------------------

    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = y[train_index], y[test_index]

    X_train_Num = X_Num[train_index]
    Y_train_Num = y_Num[train_index]
    Y_test_Num = y_Num[test_index]


    # ========================================================
    # 2. Standard RACER baseline
    # ========================================================

    racer = ROPAC(
        alpha=maxalpha,
        gamma=0.6,
        suppress_warnings=True
    )

    racer.fit(X_train, Y_train)
    racer.reduceRules()

    baseline_score = racer.score(X_test, Y_test)

    racerAvgAcc += baseline_score

    print(
        f"\nStandard RACER Fold Accuracy: "
        f"{baseline_score:.4f}"
    )


    # ========================================================
    # 3. Calculate EVERY weighting method ONCE
    # ========================================================
    #
    # IMPORTANT:
    # Each weighting method is calculated exactly once.
    # The resulting vectors are stored in the `weights`
    # dictionary and reused for every ordering method.
    #
    # No ordering is performed here.
    # No test data is used here.
    #
    # ========================================================

    weights = {}

    weights["harmfulness"] = harmfulness(
        X_train_Num,
        Y_train_Num
    )

    weights["usefulness"] = usefulness(
        X_train_Num,
        Y_train_Num
    )

    weights["LSR"] = LSR(
        X_train_Num,
        Y_train_Num
    )

    weights["LSC"] = LSC(
        X_train_Num,
        Y_train_Num
    )

    weights["N2"] = N2(
        X_train_Num,
        Y_train_Num
    )

    weights["F1"] = F1(
        X_train_Num,
        Y_train_Num
    )

    weights["mahalanobis_weights"] = mahalanobis_weights(
        X_train_Num
    )

    weights["gaussian_class_margin"] = gaussian_class_margin(
        X_train_Num,
        Y_train_Num
    )

    weights["gaussian_class_likelihood"] = gaussian_class_likelihood(
        X_train_Num,
        Y_train_Num
    )

    weights["knn_disagreeing"] = knn_disagreeing(
        X_train_Num,
        Y_train_Num
    )

    weights["knn_weighting"] = knn_weighting(
        X_train_Num,
        Y_train_Num
    )

    weights["weak_strong"] = weak_strong(
        X_train,
        Y_train,
        maxalpha,
        minalpha
    )


    # --------------------------------------------------------
    # Weighting methods that require a RACER instance
    #
    # This is intentionally kept separate from the baseline
    # RACER because your original code uses:
    #
    #     alpha = maxalpha - 0.15
    #
    # for these weighting methods.
    # --------------------------------------------------------

    weighting_racer = ROPAC(
        alpha=maxalpha - 0.15,
        gamma=0.6,
        suppress_warnings=True
    )

    weighting_racer.fit(X_train, Y_train)
    weighting_racer.reduceRules()


    weights["tree_depth"] = tree_depth(
        X_train,
        weighting_racer._final_rules_if
    )

    weights["DS_fitness"] = DS_fitness(
        X_train,
        weighting_racer._final_rules_if,
        weighting_racer._fitnesses
    )

    weights["DCP"] = DCP(
        X_train,
        Y_train,
        weighting_racer._final_rules_if,
        weighting_racer._final_rules_then
    )

    weights["DS"] = DS(
        X_train,
        weighting_racer._final_rules_if
    )


    # ========================================================
    # 4. Apply EVERY ordering method to EVERY weight vector
    # ========================================================

    fold_results = {}
    fold_predictions = {}

    for weighting_name, weight_vector in weights.items():

        # Separate dictionaries for this weighting method
        fold_results[weighting_name] = {}
        fold_predictions[weighting_name] = {}

        print(
            f"\n{'-' * 12} "
            f"Weighting: {weighting_name} "
            f"{'-' * 12}"
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # `weight_vector` was calculated ONCE above.
        #
        # Every ordering method receives this SAME vector.
        # ----------------------------------------------------

        for ordering_name, ordering_func in ordering_methods.items():

            # ------------------------------------------------
            # Create ordering from the current weighting vector
            # ------------------------------------------------

            idx = ordering_func(weight_vector)

            # ------------------------------------------------
            # Reorder ONLY training data
            # ------------------------------------------------

            X_train_sorted = X_train[idx]
            Y_train_sorted = Y_train[idx]

            # for weighted racer
            weights_sorted = weight_vector[idx]

            # ------------------------------------------------
            # RACER using the reordered training data
            # ------------------------------------------------

            racer_reordered = ROPAC(
                alpha=maxalpha,
                gamma=0.6,
                suppress_warnings=True
            )

            racer_reordered.fit(
                X_train_sorted,
                Y_train_sorted
            )

            racer_reordered.reduceRules()

            # ------------------------------------------------
            # Predict on ORIGINAL untouched X_test
            # ------------------------------------------------

            y_pred = racer_reordered.predict(X_test)

            reordered_score = racer_reordered.score(
                X_test,
                Y_test
            )

            # ------------------------------------------------
            # Save result in two independent dimensions:
            #
            # weighting_name
            #       └── ordering_name
            #
            # ------------------------------------------------

            fold_results[weighting_name][ordering_name] = (
                reordered_score
            )

            fold_predictions[weighting_name][ordering_name] = (
                np.asarray(y_pred)
            )

            # ------------------------------------------------
            # Add to cross-fold reporting structure
            # ------------------------------------------------

            fold_accuracies[ordering_name][weighting_name].append(
                reordered_score
            )

            print(
                f"Weighting: {weighting_name:<25} "
                f"| Ordering: {ordering_name:<25} "
                f"| Accuracy: {reordered_score:.4f}"
            )


    # ========================================================
    # 5. Save EVERYTHING for this fold
    # ========================================================

    fold_data = {

        # ----------------------------------------------------
        # Fold information
        # ----------------------------------------------------

        "fold": fold_number,

        "train_index": np.asarray(train_index),
        "test_index": np.asarray(test_index),


        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        "labels": {
            "Y_train": np.asarray(Y_train),
            "Y_test": np.asarray(Y_test),
            "Y_train_Num": np.asarray(Y_train_Num),
            "Y_test_Num": np.asarray(Y_test_Num)
        },


        # ----------------------------------------------------
        # All weighting vectors
        #
        # Each vector is calculated once and preserved here.
        # ----------------------------------------------------

        "weights": {
            weighting_name: np.asarray(weight_vector)
            for weighting_name, weight_vector in weights.items()
        },


        # ----------------------------------------------------
        # Accuracy:
        #
        # results[weighting][ordering]
        # ----------------------------------------------------

        "results": fold_results,


        # ----------------------------------------------------
        # Predictions:
        #
        # predictions[weighting][ordering]
        # ----------------------------------------------------

        "predictions": fold_predictions,


        # ----------------------------------------------------
        # Standard RACER baseline
        # ----------------------------------------------------

        "baseline": {
            "accuracy": float(baseline_score)
        },


        # ----------------------------------------------------
        # Parameters needed to reproduce/interpret the fold
        # ----------------------------------------------------

        "parameters": {
            "maxalpha": maxalpha,
            "minalpha": minalpha,
            "gamma": 0.6,
            "weighting_racer_alpha": maxalpha - 0.15,
            "tree_depth_alpha": maxalpha - 0.15,
            "DS_fitness_alpha": maxalpha - 0.15,
            "DCP_alpha": maxalpha - 0.15,
            "DS_alpha": maxalpha - 0.15,
            "baseline_racer_alpha": maxalpha,
            "reordered_racer_alpha": maxalpha,
            "suppress_warnings": True
        },


        # ----------------------------------------------------
        # Explicit method names
        # ----------------------------------------------------

        "weighting_methods": list(weights.keys()),
        "ordering_methods": list(ordering_methods.keys()),


        # ----------------------------------------------------
        # Dataset information
        # ----------------------------------------------------

        "dataset_name": datasetname
    }


    # --------------------------------------------------------
    # Save one file per fold
    # --------------------------------------------------------

    fold_path = os.path.join(
        save_dir,
        f"fold_{fold_number:02d}.joblib"
    )

    joblib.dump(
        fold_data,
        fold_path
    )

    print(
        f"\nSaved fold {fold_number} results to:"
        f"\n{fold_path}"
    )


# ============================================================
# 6. Final Reporting
# ============================================================

print(f"\n{'=' * 17} Standard RACER {'=' * 17}")

baseline_mean = racerAvgAcc / kf.get_n_splits()

print(
    f"Standard RACER: "
    f"{baseline_mean * 100:.2f}%"
)


# ============================================================
# Summarize ordering methods SEPARATELY for each weighting
# ============================================================

final_summary = {
    "baseline": {
        "mean_accuracy": float(baseline_mean)
    },
    "weighting_results": {}
}


# Get weighting method names
weighting_names = list(
    next(iter(fold_accuracies.values())).keys()
)


for weighting_name in weighting_names:

    print(
        f"\n{'=' * 17} "
        f"Weighting: {weighting_name} "
        f"{'=' * 17}"
    )

    # Collect the fold accuracies for every ordering
    weighting_scores = {}

    for ordering_name in ordering_methods:

        weighting_scores[ordering_name] = (
            fold_accuracies[ordering_name][weighting_name]
        )

    # Average over folds for each ordering
    weighting_summary = summarize_results(
        weighting_scores
    )

    final_summary["weighting_results"][weighting_name] = (
        weighting_summary
    )


In [36]:
# ============================================================
# Save final summary to Excel
# ============================================================

excel_path = os.path.join(
    save_dir,
    "final_summary.xlsx"
)


# ------------------------------------------------------------
# Create table:
# Rows    = weighting methods
# Columns = ordering methods
# Values  = average accuracy over folds
# ------------------------------------------------------------

accuracy_table = {}

for weighting_name, ordering_results in (
    final_summary["weighting_results"].items()
):

    accuracy_table[weighting_name] = {}

    for ordering_name, result in ordering_results.items():

        accuracy_table[weighting_name][ordering_name] = (
            result["mean_accuracy"]
        )


accuracy_df = pd.DataFrame.from_dict(
    accuracy_table,
    orient="index"
)

accuracy_df.index.name = "Weighting Method"
accuracy_df = accuracy_df.reset_index()

# ------------------------------------------------------------
# Save Excel
# ------------------------------------------------------------

sheet_name = f"RACER Accuracy ({baseline_mean * 100:.2f}%)"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        index=False,
        startrow=0
    )


print(
    f"\nFinal summary saved to:"
    f"\n{excel_path}"
)


Final summary saved to:
../saved_results\31-dermatology\final_summary.xlsx


In [37]:
def create_builtin_models(random_state=42):
    """
    Built-in classifiers available in scikit-learn.

    X must be numeric.
    y should be a one-dimensional vector of class labels.
    """
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1
        ),

        "MLP": make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(100,),
                max_iter=1000,
                random_state=random_state
            )
        ),

        "Bagging": BaggingClassifier(
            estimator=DecisionTreeClassifier(
                random_state=random_state
            ),
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1
        ),

        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=1.0,
                random_state=random_state
            )
        ),

        "CART": DecisionTreeClassifier(
            criterion="gini",
            random_state=random_state
        ),

        "IBk": make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(
                n_neighbors=5
            )
        ),

        "Naive Bayes": GaussianNB()
    }

In [38]:
comparison_save_dir = os.path.join(
    "../saved_results",
    datasetname,
    "algorithm_comparison"
)

# ============================================================
# Algorithms to compare
# ============================================================

ropac_algorithms = {
    "RACER": Algorithm.RACER,
    "RACERS": Algorithm.RACERS,
    "RRa": Algorithm.RRA,
    "RRd": Algorithm.RRD,
    "RRa,s": Algorithm.RRA_S,
    "RRd,s": Algorithm.RRD_S,
    "ROPAC-M": Algorithm.ROPAC_M,
    "ROPAC-L": Algorithm.ROPAC_L,
}


builtin_models = create_builtin_models(random_state=42)


# ============================================================
# Storage
#
# comparison_fold_accuracies[algorithm] = [
#     fold 1 accuracy,
#     fold 2 accuracy,
#     ...
# ]
# ============================================================

comparison_fold_accuracies = {
    name: []
    for name in (
        list(ropac_algorithms.keys())
        + list(builtin_models.keys())
    )
}


# Save predictions separately
#
# predictions[algorithm][fold_number] = y_pred
#
comparison_predictions = {
    name: {}
    for name in comparison_fold_accuracies
}


# ============================================================
# Save comparison results
# ============================================================

os.makedirs(
    comparison_save_dir,
    exist_ok=True
)


# ============================================================
# Use the SAME saved fold files
# ============================================================

n_folds = kf.get_n_splits()


for fold_number in range(1, n_folds + 1):

    print(
        f"\n{'=' * 17} "
        f"Comparison Fold {fold_number} "
        f"{'=' * 17}"
    )


    # --------------------------------------------------------
    # Load EXACT train/test split used in the original
    # experiment
    # --------------------------------------------------------

    original_fold_path = os.path.join(
        save_dir,
        f"fold_{fold_number:02d}.joblib"
    )

    original_fold_data = joblib.load(
        original_fold_path
    )


    train_index = np.asarray(
        original_fold_data["train_index"]
    )

    test_index = np.asarray(
        original_fold_data["test_index"]
    )


    # --------------------------------------------------------
    # Recreate EXACT same split
    # --------------------------------------------------------

    X_train = X[train_index]
    X_test = X[test_index]

    Y_train = y[train_index]
    Y_test = y[test_index]

    X_train_Num = X_Num[train_index]
    X_test_Num = X_Num[test_index]

    Y_train_Num = y_Num[train_index]
    Y_test_Num = y_Num[test_index]


    # ========================================================
    # 1. ROPAC algorithms
    # ========================================================

    for algorithm_name, algorithm in ropac_algorithms.items():

        print(
            f"\n{algorithm_name}"
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Use the SAME parameters for every ROPAC algorithm
        # ----------------------------------------------------

        ropac = ROPAC(
            alpha=maxalpha,
            gamma=0.6,
            suppress_warnings=True
        )

        ropac.fit(
            X_train,
            Y_train,
            algorithm=algorithm
        )

        ropac.reduceRules()

        y_pred = ropac.predict(
            X_test
        )

        accuracy = ropac.score(
            X_test,
            Y_test
        )


        comparison_fold_accuracies[
            algorithm_name
        ].append(
            accuracy
        )

        comparison_predictions[
            algorithm_name
        ][fold_number] = np.asarray(
            y_pred
        )


        print(
            f"{algorithm_name:<15}: "
            f"{accuracy:.4f}"
        )


    # ========================================================
    # 2. Built-in scikit-learn algorithms
    # ========================================================

    for algorithm_name, model in builtin_models.items():

        print(
            f"\n{algorithm_name}"
        )


        # ----------------------------------------------------
        # Clone the model so every fold starts fresh.
        # ----------------------------------------------------

        model_fold = clone(model)


        # ----------------------------------------------------
        # Built-in models use the NUMERIC representation.
        #
        # This avoids feeding categorical/string values into
        # scikit-learn classifiers.
        # ----------------------------------------------------

        model_fold.fit(
            X_train_Num,
            Y_train_Num
        )


        y_pred = model_fold.predict(
            X_test_Num
        )


        accuracy = np.mean(
            y_pred == Y_test_Num
        )


        comparison_fold_accuracies[
            algorithm_name
        ].append(
            accuracy
        )

        comparison_predictions[
            algorithm_name
        ][fold_number] = np.asarray(
            y_pred
        )


        print(
            f"{algorithm_name:<15}: "
            f"{accuracy:.4f}"
        )


    # ========================================================
    # 3. Save this comparison fold
    # ========================================================

    fold_comparison_data = {

        "fold": fold_number,

        "train_index": train_index,
        "test_index": test_index,

        "labels": {
            "Y_train": np.asarray(Y_train),
            "Y_test": np.asarray(Y_test),
            "Y_train_Num": np.asarray(Y_train_Num),
            "Y_test_Num": np.asarray(Y_test_Num)
        },

        "results": {
            algorithm_name: {
                "accuracy":
                    comparison_fold_accuracies[
                        algorithm_name
                    ][fold_number - 1]
            }
            for algorithm_name
            in comparison_fold_accuracies
        },

        "predictions": {
            algorithm_name:
                comparison_predictions[
                    algorithm_name
                ][fold_number]
            for algorithm_name
            in comparison_predictions
        },

        "algorithms": {
            "ROPAC": list(
                ropac_algorithms.keys()
            ),
            "built_in": list(
                builtin_models.keys()
            )
        },

        "parameters": {
            "maxalpha": maxalpha,
            "minalpha": minalpha,
            "gamma": 0.6,
            "suppress_warnings": True,
            "random_state": 42
        },

        "dataset_name": datasetname
    }


    fold_comparison_path = os.path.join(
        comparison_save_dir,
        f"fold_{fold_number:02d}.joblib"
    )


    joblib.dump(
        fold_comparison_data,
        fold_comparison_path
    )




    print(
        f"\nSaved comparison fold:"
        f"\n{fold_comparison_path}"
    )

# ------------------------------------------------------------
# Save Excel
# ------------------------------------------------------------
comparison_summary = summarize_results(
    comparison_fold_accuracies
)

comparison_excel_path = os.path.join(
    comparison_save_dir,
    "algorithm_comparison.xlsx"
)

comparison_df = pd.DataFrame({
    "Algorithm": list(comparison_summary.keys()),
    "Average Accuracy": [
        comparison_summary[name]["mean_accuracy"]
        for name in comparison_summary
    ]
})

comparison_df = comparison_df.set_index(
    "Algorithm"
).T

with pd.ExcelWriter(
    comparison_excel_path,
    engine="openpyxl"
) as writer:

    comparison_df.to_excel(
        writer,
        sheet_name="Results",
        index=False,
        startrow=0
    )

print(
    f"\nAlgorithm comparison Excel saved to:"
    f"\n{comparison_excel_path}"
)


================= Comparison Fold 1 =================

RACER
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
RACER          : 0.9730

RACERS
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
RACERS         : 0.9730

RRa
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
RRa            : 0.9459

RRd
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
RRd            : 0.9459

RRa,s
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
RRa,s          : 0.9459

RRd,s
Labels for these instances

# Other measures

In [39]:
# ============================================================
# Choose measure
# ============================================================

measure_name = "f1"

# Supported:
# "accuracy"
# "f1"
# "precision"
# "recall"


# ============================================================
# Measure calculation
# ============================================================

def calculate_measure(y_true, y_pred, measure_name):

    y_true = labels_1d(y_true)
    y_pred = labels_1d(y_pred)

    if measure_name == "accuracy":

        return accuracy_score(
            y_true,
            y_pred
        )

    elif measure_name == "f1":

        return f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )

    elif measure_name == "precision":

        return precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )

    elif measure_name == "recall":

        return recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )

    else:

        raise ValueError(
            f"Unsupported measure: {measure_name}"
        )


# ============================================================
# Paths
# ============================================================

comparison_save_dir = os.path.join(
    "../saved_results",
    datasetname,
    "algorithm_comparison"
)

weighting_save_dir = os.path.join(
    "../saved_results",
    datasetname
)


# ============================================================
# Helper
# ============================================================

def labels_1d(y):
    y = np.asarray(y)

    return (
        np.argmax(y, axis=1)
        if y.ndim == 2
        else y
    )


# ============================================================
# Get fold files
# ============================================================

comparison_folds = sorted([
    f for f in os.listdir(comparison_save_dir)
    if f.startswith("fold_") and f.endswith(".joblib")
])

weighting_folds = sorted([
    f for f in os.listdir(weighting_save_dir)
    if f.startswith("fold_") and f.endswith(".joblib")
])


# ============================================================
# 1. Algorithm Comparison
# ============================================================

first_comparison = joblib.load(
    os.path.join(
        comparison_save_dir,
        comparison_folds[0]
    )
)

algorithm_names = list(
    first_comparison["predictions"].keys()
)

ropac_algorithms = set(
    first_comparison["algorithms"]["ROPAC"]
)


measure_comparison = {
    name: []
    for name in algorithm_names
}


for fold_file in comparison_folds:

    data = joblib.load(
        os.path.join(
            comparison_save_dir,
            fold_file
        )
    )

    for name in algorithm_names:

        if name in ropac_algorithms:
            y_true = data["labels"]["Y_test"]
        else:
            y_true = data["labels"]["Y_test_Num"]

        y_pred = data["predictions"][name]

        score = calculate_measure(
            y_true,
            y_pred,
            measure_name
        )

        measure_comparison[name].append(score)


# ============================================================
# Average over folds
# ============================================================

measure_comparison_avg = {
    name: np.mean(scores)
    for name, scores
    in measure_comparison.items()
}


# ============================================================
# Save Algorithm Comparison
# ============================================================

comparison_df = pd.DataFrame(
    [measure_comparison_avg],
    index=[f"Average {measure_name.capitalize()}"]
)


comparison_path = os.path.join(
    comparison_save_dir,
    f"algorithm_{measure_name}.xlsx"
)


with pd.ExcelWriter(
    comparison_path,
    engine="openpyxl"
) as writer:

    comparison_df.to_excel(
        writer,
        sheet_name=measure_name.capitalize()
    )


# ============================================================
# 2. Weighting × Ordering
# ============================================================

first_weighting = joblib.load(
    os.path.join(
        weighting_save_dir,
        weighting_folds[0]
    )
)


measure_weighting = {
    weighting: {
        ordering: []
        for ordering in orderings
    }
    for weighting, orderings
    in first_weighting["predictions"].items()
}


for fold_file in weighting_folds:

    data = joblib.load(
        os.path.join(
            weighting_save_dir,
            fold_file
        )
    )

    y_true = data["labels"]["Y_test"]

    for weighting, orderings in (
        data["predictions"].items()
    ):

        for ordering, y_pred in (
            orderings.items()
        ):

            score = calculate_measure(
                y_true,
                y_pred,
                measure_name
            )

            measure_weighting[
                weighting
            ][ordering].append(score)


# ============================================================
# Average over folds
# ============================================================

measure_table = {}

for weighting, orderings in (
    measure_weighting.items()
):

    measure_table[weighting] = {
        ordering: np.mean(scores)
        for ordering, scores
        in orderings.items()
    }


measure_df = pd.DataFrame.from_dict(
    measure_table,
    orient="index"
)

measure_df.index.name = "Weighting Method"


# ============================================================
# Standard RACER Measure
# ============================================================

racer_measure_avg = (
    measure_comparison_avg["RACER"]
)


# ============================================================
# Sheet name
# ============================================================

sheet_name = (
    f"RACER {measure_name.capitalize()} "
    f"({racer_measure_avg * 100:.2f}%)"
)


# ============================================================
# Save Weighting × Ordering
# ============================================================

weighting_path = os.path.join(
    weighting_save_dir,
    f"weighting_ordering_{measure_name}.xlsx"
)


with pd.ExcelWriter(
    weighting_path,
    engine="openpyxl"
) as writer:

    measure_df.to_excel(
        writer,
        sheet_name=sheet_name
    )


# ============================================================
# Output
# ============================================================

print(
    f"\n{measure_name.capitalize()} for algorithm comparison saved to:"
    f"\n{comparison_path}"
)

print(
    f"\n{measure_name.capitalize()} for weighting × ordering saved to:"
    f"\n{weighting_path}"
)


F1 for algorithm comparison saved to:
../saved_results\31-dermatology\algorithm_comparison\algorithm_f1.xlsx

F1 for weighting × ordering saved to:
../saved_results\31-dermatology\weighting_ordering_f1.xlsx
